## 


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/02_llm.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Lab 2: Comparing Ollama Models</h2>
            <h4 style="color:#00bf00;">© 2026 Dr Elena Kramer, Braude College, Israel.</h4>
            <span style="color:#00bfff;">In this lab, we send the same question to several models on the configured Ollama server, collect their answers, and use a judge model to rank them.

The notebook uses the OpenAI-compatible Python client to connect to Ollama. Set `OLLAMA_BASE_URL` in the project-root `.env` file; the model names are defined in the notebook code.
            </span>
        </td>
    </tr>
</table>

In [29]:
# Lab 2 — Ollama Model Comparison
# Original additions and modifications © 2026 Dr Elena Kramer, Braude College, Israel.
# Adapted course content and third-party materials remain with their respective rights holders.

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

load_dotenv(override=True)

True

In [30]:
# Always remember to do this!
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")
if not OLLAMA_BASE_URL:
    raise ValueError("Set OLLAMA_BASE_URL in the project-root .env file")

openai = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama"
)

In [31]:
MODEL_NAMES = [
    "gpt-oss:120b",
    "qwen3.5:122b",
    "qwen3.8:27b",
    "gpt-oss:20b",
    "llama3.1:8b",
]
QUESTION_MODEL = "gpt-oss:120b"
JUDGE_MODEL = "gpt-oss:120b"

These are Ollama model tags on the configured server. The OpenAI Python package is used only as a compatible client; requests go to the URL in `OLLAMA_BASE_URL`.

In [51]:
request = """
Please come up with a challenging, nuanced question with a succinct answer,
that I can ask a number of LLMs to evaluate their intelligence.
Not a mathematical puzzle, but more of a thought-provoking question that requires intelligent insight.
Include in your question that the answer must be short.
"""
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]


In [52]:
messages

[{'role': 'user',
  'content': '\nPlease come up with a challenging, nuanced question with a succinct answer,\nthat I can ask a number of LLMs to evaluate their intelligence.\nNot a mathematical puzzle, but more of a thought-provoking question that requires intelligent insight.\nInclude in your question that the answer must be short.\nAnswer only with the question, no explanation.'}]

In [53]:
response = openai.chat.completions.create(model=QUESTION_MODEL, messages=messages)
question = response.choices[0].message.content
display(Markdown(question))

Explain in no more than five words the core distinction between a model that predicts language and a mind that truly comprehends it.

In [54]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

In [55]:
def record(model_name, answer):
    competitors.append(model_name)
    answers.append(answer)
    display(Markdown(answer))

In [56]:
model_name = MODEL_NAMES[0]
response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content
record(model_name, answer)

Predicts patterns, doesn't understand meaning.

In [16]:
# model_name = MODEL_NAMES[1]
# response = openai.chat.completions.create(model=model_name, messages=messages)
# answer = response.choices[0].message.content
# record(model_name, answer)

In [57]:
model_name = MODEL_NAMES[2]
response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content
record(model_name, answer)

Prediction versus lived experience.

In [58]:
model_name = MODEL_NAMES[3]
response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content
record(model_name, answer)

Pattern matching versus genuine understanding.

In [59]:
model_name = MODEL_NAMES[4]
response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content
record(model_name, answer)

Predicts vs truly interprets language.

In [60]:
# So where are we?

print(len(competitors))
print(competitors)
print(answers)

4
['gpt-oss:120b', 'qwen3.8:27b', 'gpt-oss:20b', 'llama3.1:8b']
["Predicts patterns, doesn't understand meaning.", 'Prediction versus lived experience.', 'Pattern matching versus genuine understanding.', 'Predicts vs truly interprets language.']


In [61]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")


Competitor: gpt-oss:120b

Predicts patterns, doesn't understand meaning.
Competitor: qwen3.8:27b

Prediction versus lived experience.
Competitor: gpt-oss:20b

Pattern matching versus genuine understanding.
Competitor: llama3.1:8b

Predicts vs truly interprets language.


In [62]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [63]:
print(together)

# Response from competitor 1

Predicts patterns, doesn't understand meaning.

# Response from competitor 2

Prediction versus lived experience.

# Response from competitor 3

Pattern matching versus genuine understanding.

# Response from competitor 4

Predicts vs truly interprets language.




In [64]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [65]:
print(judge)

You are judging a competition between 4 competitors.
Each model has been given this question:

Explain in no more than five words the core distinction between a model that predicts language and a mind that truly comprehends it.

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}

Here are the responses from each competitor:

# Response from competitor 1

Predicts patterns, doesn't understand meaning.

# Response from competitor 2

Prediction versus lived experience.

# Response from competitor 3

Pattern matching versus genuine understanding.

# Response from competitor 4

Predicts vs truly interprets language.



Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks.


In [66]:
judge_messages = [{"role": "user", "content": judge}]

In [67]:
response = openai.chat.completions.create(model=JUDGE_MODEL, messages=judge_messages)
results = response.choices[0].message.content
print(results)

{"results": ["1", "3", "4", "2"]}


In [49]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")

Rank 1: gpt-oss:120b
Rank 2: qwen3.8:27b
Rank 3: gpt-oss:20b
Rank 4: llama3.1:8b
